# LEDGAR Clause Classification Pipeline

Coursework support notebook for contract clause classification using LexGLUE LEDGAR.

Academic boundary: this notebook prepares code, metrics, plots, predictions, and saved outputs only. It does not write report text, invent results, or provide final conclusions.


## 1. Setup, Colab Bootstrap, and Imports

Run this cell first. In Colab, edit the TODO path placeholders if your repo is stored in Google Drive or a custom `/content` folder.


In [ ]:
from pathlib import Path
import importlib
import importlib.util
import json
import os
import subprocess
import sys

RANDOM_STATE = 42

IN_COLAB = bool(os.getenv("COLAB_RELEASE_TAG")) or importlib.util.find_spec("google.colab") is not None

# TODO for Google Colab:
# 1. If your repo is in Google Drive, set MOUNT_GOOGLE_DRIVE = True.
# 2. Set COLAB_PROJECT_ROOT_OVERRIDE to the folder containing pyproject.toml, src/, and modules/.
# 3. If you clone into /content/Natural-Language-Processing, you can leave the override as None.
#
# Example clone cell to run before this one if needed:
# !git clone https://github.com/YOUR_USERNAME/Natural-Language-Processing.git /content/Natural-Language-Processing
MOUNT_GOOGLE_DRIVE = False
COLAB_PROJECT_ROOT_OVERRIDE = None
# COLAB_PROJECT_ROOT_OVERRIDE = "/content/drive/MyDrive/path/to/Natural-Language-Processing"  # TODO: edit for your Drive
# COLAB_PROJECT_ROOT_OVERRIDE = "/content/Natural-Language-Processing"  # TODO: edit if your clone path differs

# TODO for local Jupyter only: set this if Jupyter was started outside the repo.
LOCAL_PROJECT_ROOT_OVERRIDE = None

# Set this False after the first successful Colab install if rerunning cells in the same runtime.
INSTALL_REQUIREMENTS_IN_COLAB = True

# Optional W&B settings. Do not paste API keys into committed notebooks.
os.environ.setdefault("WANDB_PROJECT", "ledgar-clause-classification")
# os.environ["WANDB_ENTITY"] = "your-wandb-entity"  # TODO: optional W&B username/team
# os.environ["WANDB_API_KEY"] = "paste-key-in-runtime-only"  # TODO: prefer Colab secrets or wandb.login()

if IN_COLAB and MOUNT_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")

PROJECT_ROOT_OVERRIDE = COLAB_PROJECT_ROOT_OVERRIDE if IN_COLAB else LOCAL_PROJECT_ROOT_OVERRIDE
KNOWN_LOCAL_PROJECT_ROOT = Path(r"C:\Users\ybenj\Documents\GitHub\Education\NLP\Natural-Language-Processing")


def looks_like_project_root(candidate: Path) -> bool:
    return (
        (candidate / "pyproject.toml").exists()
        and (candidate / "src").exists()
        and (candidate / "modules").exists()
    )


def find_project_root(start: Path) -> Path:
    candidates = []
    if PROJECT_ROOT_OVERRIDE:
        candidates.append(Path(PROJECT_ROOT_OVERRIDE))
    if os.getenv("LEDGAR_PROJECT_ROOT"):
        candidates.append(Path(os.environ["LEDGAR_PROJECT_ROOT"]))
    candidates.extend([start, *start.parents])

    if IN_COLAB:
        candidates.extend([
            Path("/content/Natural-Language-Processing"),
            Path("/content/drive/MyDrive/Natural-Language-Processing"),
        ])
    else:
        candidates.append(KNOWN_LOCAL_PROJECT_ROOT)

    seen = set()
    for candidate in candidates:
        candidate = candidate.expanduser().resolve()
        if candidate in seen:
            continue
        seen.add(candidate)
        if looks_like_project_root(candidate):
            return candidate

    raise FileNotFoundError(
        "Could not find the project root containing pyproject.toml, src/, and modules/. "
        "In Colab, clone/upload the repo, then edit COLAB_PROJECT_ROOT_OVERRIDE above."
    )


PROJECT_ROOT = find_project_root(Path.cwd())
os.environ["LEDGAR_PROJECT_ROOT"] = str(PROJECT_ROOT)
SRC_DIR = PROJECT_ROOT / "src"
for import_path in (PROJECT_ROOT, SRC_DIR):
    if str(import_path) not in sys.path:
        sys.path.insert(0, str(import_path))

if IN_COLAB and INSTALL_REQUIREMENTS_IN_COLAB:
    requirements_file = PROJECT_ROOT / "requirements-colab.txt"
    if not requirements_file.exists():
        requirements_file = PROJECT_ROOT / "requirements.txt"
    print(f"Installing Colab dependencies from: {requirements_file}")
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "-q", "-r", str(requirements_file)],
        cwd=str(PROJECT_ROOT),
    )
    importlib.invalidate_caches()

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from IPython.display import display

np.random.seed(RANDOM_STATE)

print(f"Colab runtime: {IN_COLAB}")
print(f"Kernel Python: {sys.executable}")
print(f"Working directory: {Path.cwd().resolve()}")
print(f"Project root: {PROJECT_ROOT}")
print(f"src on sys.path: {str(SRC_DIR) in sys.path}")

try:
    from datasets import load_dataset
    print("datasets package available")
except Exception as exc:
    print(f"WARNING: datasets import failed: {type(exc).__name__}: {exc}")

TRANSFORMERS_AVAILABLE = importlib.util.find_spec("transformers") is not None
if TRANSFORMERS_AVAILABLE:
    import transformers
    print(f"transformers package available: {transformers.__version__}")
else:
    print("transformers package is not installed. Transformer cells are guarded and disabled by default.")

try:
    from modules.preprocess import (
        create_eda_outputs,
        create_label_mapping,
        dataset_overview,
        ensure_project_dirs,
        export_raw_hf_splits,
        filter_splits_to_labels,
        load_ledgar_dataset,
        load_processed_splits,
        print_filtered_summary,
        print_label_inspection,
        save_label_artifacts,
        save_processed_splits,
        save_selection_artifacts,
        select_labels,
        standardise_splits,
        warn_messages,
        apply_label_mapping,
    )
    from modules.train_classical import (
        create_error_analysis,
        create_final_comparison,
        load_label_mapping,
        run_baselines,
        run_classical_models,
    )
    from modules.train_transformer import run_transformer_experiments, transformers_available
    from modules.config import get_config
    from modules.wandb_tracking import (
        log_preprocessing_run,
        login_wandb_if_needed,
        verify_wandb_auth,
        wandb_is_installed,
    )
except ModuleNotFoundError as exc:
    raise ModuleNotFoundError(
        f"{exc}\n\nLocal project modules were not importable. "
        f"Project root detected as: {PROJECT_ROOT}\n"
        "Local PowerShell install:\n"
        f"cd \"{PROJECT_ROOT}\"\n"
        "python -m pip install -r requirements.txt\n\n"
        "Colab install from the repo root:\n"
        "%pip install -q -r requirements-colab.txt\n"
        "Then restart/rerun this setup cell."
    ) from exc

CONFIG = get_config(PROJECT_ROOT)
CONFIG.paths.ensure_dirs()
paths = ensure_project_dirs(PROJECT_ROOT)
for name, path in paths.items():
    if name != "root":
        print(f"{name}: {path}")


## 1.1 Experiment Flags

These flags keep LEDGAR as the main experiment and make CUAD an optional secondary branch. Paths are relative to the project root detected above.

In [ ]:
SEED = RANDOM_STATE

RUN_CUAD_EXPERIMENT = True
CUAD_DATA_PATH = "data/raw/cuad/"
CUAD_OUTPUT_DIR = "data/processed/cuad/"
CUAD_MIN_SAMPLES_PER_CLASS = 10
CUAD_TOP_K_LABELS = 20
CUAD_RESULTS_DIR = "results/cuad/"

print(f"RUN_CUAD_EXPERIMENT: {RUN_CUAD_EXPERIMENT}")
print(f"CUAD raw path: {PROJECT_ROOT / CUAD_DATA_PATH}")
print(f"CUAD processed output path: {PROJECT_ROOT / CUAD_OUTPUT_DIR}")
print(f"CUAD results path: {PROJECT_ROOT / CUAD_RESULTS_DIR}")

## 2. GPU / CUDA Check

CUDA is only relevant for transformer fine-tuning. TF-IDF, Logistic Regression, and Linear SVM are CPU-based.


In [ ]:
print(f"torch.cuda.is_available(): {torch.cuda.is_available()}")
print(f"torch.version.cuda: {torch.version.cuda}")
print(f"torch.cuda.device_count(): {torch.cuda.device_count()}")

if torch.cuda.is_available():
    print(f"torch.cuda.get_device_name(0): {torch.cuda.get_device_name(0)}")
    DEVICE = "cuda"
else:
    DEVICE = "cpu"
    print("WARNING: CUDA is unavailable. Transformer fine-tuning will run on CPU if enabled and may be slow.")

print(f"Selected device: {DEVICE}")
print("TF-IDF, Logistic Regression, and Linear SVM are CPU-based.")
print("CUDA mainly matters for BERT/LegalBERT transformer fine-tuning.")


## 3. W&B Tracking Check

W&B logging is compulsory for this pipeline. In Colab, this cell can prompt for your W&B API key through `wandb.login()` if `WANDB_API_KEY` is not already configured.

TODO: if you use a W&B team, set `WANDB_ENTITY` in the setup cell before running this check.

Do not commit API keys to this notebook or repository.


In [ ]:
print(f"wandb installed: {wandb_is_installed()}")
print(f"W&B project: {CONFIG.wandb.project}")
print(f"W&B entity: {CONFIG.wandb.entity}")
print(f"W&B mode: {CONFIG.wandb.mode}")

if IN_COLAB:
    print("Colab detected: wandb.login() may prompt for your API key if one is not already configured.")
    login_wandb_if_needed(CONFIG.wandb)
else:
    verify_wandb_auth(CONFIG.wandb)

print("W&B is required, online mode is enforced, and authentication was verified.")


## 4. Reference Notebook Style Check

Inspect local reference notebooks only for style signals. Do not copy their content, results, or exercises.


In [ ]:
reference_dir = paths["reference_notebooks"]
style_rows = []

if reference_dir.exists():
    for notebook_path in sorted(reference_dir.glob("*.ipynb")):
        with notebook_path.open("r", encoding="utf-8") as f:
            nb = json.load(f)
        notebook_cells = nb.get("cells", [])
        markdown_cells = [cell for cell in notebook_cells if cell.get("cell_type") == "markdown"]
        code_cells = [cell for cell in notebook_cells if cell.get("cell_type") == "code"]
        heading_count = 0
        todo_count = 0
        code_lengths = []
        for cell in markdown_cells:
            source = "".join(cell.get("source", []))
            todo_count += source.lower().count("todo")
            heading_count += sum(1 for line in source.splitlines() if line.strip().startswith("#"))
        for cell in code_cells:
            source = "".join(cell.get("source", []))
            code_lengths.append(len([line for line in source.splitlines() if line.strip()]))
        style_rows.append({
            "notebook": notebook_path.name,
            "markdown_cells": len(markdown_cells),
            "code_cells": len(code_cells),
            "heading_count": heading_count,
            "todo_mentions": todo_count,
            "avg_code_lines": round(float(np.mean(code_lengths)), 1) if code_lengths else 0,
        })
else:
    print("No reference_notebooks directory found. Continuing without local style inspection.")

if style_rows:
    display(pd.DataFrame(style_rows))

print("Style summary for this notebook:")
print("- numbered headings")
print("- short markdown explanations")
print("- compact code cells")
print("- TODO prompts for student inspection")
print("- clear displayed tables and saved outputs")


## 5. Load LexGLUE LEDGAR

Load the main dataset from Hugging Face. If loading fails, print the error clearly and attempt the configured JSONL fallback.


In [ ]:
ds, dataset_metadata = load_ledgar_dataset(PROJECT_ROOT)
overview = dataset_overview(ds)

print(f"Dataset source: {dataset_metadata['dataset_source']}")
print(f"Available splits: {overview['splits']}")
print(f"Rows per split: {overview['rows_per_split']}")
print("Dataset features:")
print(overview["features"])

features = dataset_metadata.get("features")
label_feature = features.get("label") if hasattr(features, "get") else None
print("Label feature information:")
print(label_feature)
if hasattr(label_feature, "names"):
    print(f"Number of label names: {len(label_feature.names)}")
    print(f"First 10 label names: {label_feature.names[:10]}")

print("First 3 training examples:")
display(pd.DataFrame(overview["train_examples"]))


## 6. Export Raw Hugging Face Splits to JSONL

Save the official Hugging Face splits as raw JSONL files.


In [ ]:
raw_paths = {}
if dataset_metadata["dataset_source"] == "huggingface":
    raw_paths = export_raw_hf_splits(ds, paths["raw_lexglue"])
    for split, path in raw_paths.items():
        print(f"Saved raw {split}: {path}")
else:
    print("Fallback dataset is in use, so raw Hugging Face split export was skipped.")


## 7. Standardise Schema

Convert dataset rows to: `text`, `label`, `label_id`, `source_dataset`, `source_id`, `split`.

Preprocessing keeps legal wording intact: whitespace is normalised only.


In [ ]:
standardised_splits = standardise_splits(ds, dataset_metadata)

for split, df in standardised_splits.items():
    print(f"{split}: {df.shape}")
    print(df.columns.tolist())

display(standardised_splits["train"].head(3))


## 8. Label Inspection

Inspect label frequencies before selecting a modelling subset.

TODO: Review the top and bottom labels before changing the label-selection settings.


In [ ]:
print_label_inspection(standardised_splits)
label_artifacts = save_label_artifacts(standardised_splits, paths["outputs"])
print(label_artifacts)


## 9. Label Filtering

Default setting: use the top 20 labels by training frequency. Manual labels are matched case-insensitively if that mode is selected.


In [ ]:
LABEL_SELECTION_MODE = "top_n"  # options: "all", "top_n", "manual"
TOP_N_LABELS = 20
MIN_EXAMPLES_PER_LABEL = 30
MANUAL_SELECTED_LABELS = []

selected_labels, selection_warnings = select_labels(
    standardised_splits["train"],
    mode=LABEL_SELECTION_MODE,
    top_n=TOP_N_LABELS,
    manual_labels=MANUAL_SELECTED_LABELS,
    min_examples_per_label=MIN_EXAMPLES_PER_LABEL,
)
warn_messages(selection_warnings)

filtered_splits = filter_splits_to_labels(standardised_splits, selected_labels)
label_to_id, id_to_label = create_label_mapping(selected_labels)
processed_splits = apply_label_mapping(filtered_splits, label_to_id)
selection_artifacts = save_selection_artifacts(selected_labels, label_to_id, id_to_label, paths["outputs"])

print_filtered_summary(processed_splits, selected_labels)
print(selection_artifacts)


## 10. Save Processed Splits

Save processed train, validation, and test splits as JSONL using the standard schema.


In [ ]:
processed_paths = save_processed_splits(processed_splits, paths["processed"])
for split, path in processed_paths.items():
    print(f"Saved processed {split}: {path}")

schema_check = {split: df.columns.tolist() for split, df in processed_splits.items()}
print(schema_check)


## 11. Exploratory Data Analysis

Create simple tables and figures for class distribution and text length.

TODO: Inspect these outputs before deciding whether to adjust label filtering.


In [ ]:
eda_outputs = create_eda_outputs(processed_splits, paths["outputs"], paths["figures"])
print(eda_outputs)

combined_processed = pd.concat(processed_splits.values(), ignore_index=True)
class_distribution = (
    combined_processed.groupby(["split", "label"])
    .size()
    .reset_index(name="count")
    .sort_values(["split", "count"], ascending=[True, False])
)
display(class_distribution.head(30))

length_stats = combined_processed.assign(
    word_count=combined_processed["text"].str.split().str.len(),
    character_count=combined_processed["text"].str.len(),
)[["word_count", "character_count"]].describe()
display(length_stats)

examples = pd.read_json(paths["outputs"] / "example_clauses.jsonl", lines=True)
display(examples.head(10))

metadata_paths = {
    **selection_artifacts,
    **eda_outputs,
    "label_names": paths["outputs"] / "label_names.txt",
    "label_counts": paths["outputs"] / "label_counts.json",
}
log_preprocessing_run(
    raw_paths=raw_paths,
    processed_paths=processed_paths,
    metadata_paths=metadata_paths,
    sample_frames=processed_splits,
    config={"pipeline": CONFIG.to_dict(), "dataset_source": dataset_metadata["dataset_source"]},
    wandb_config=CONFIG.wandb,
)
print("Preprocessing data, metadata, samples, and figures logged to W&B.")


## 12. Evaluation Helpers

Primary metric: macro-F1. Secondary metrics include accuracy, weighted-F1, macro precision, macro recall, per-class F1, and confusion matrix.


In [ ]:
train_df, validation_df, test_df = load_processed_splits(paths["processed"]).values()
label_to_id, id_to_label = load_label_mapping(paths["outputs"])

print(f"Train rows: {len(train_df)}")
print(f"Validation rows: {len(validation_df)}")
print(f"Test rows: {len(test_df)}")
print(f"Labels: {len(id_to_label)}")
print("Evaluation helpers are imported from src/evaluate.py and src/train_classical.py.")


## 13. Baseline Models

Run random and majority-class baselines on validation and test splits. Test predictions are saved as JSONL.


In [ ]:
baseline_results = run_baselines(
    train_df,
    validation_df,
    test_df,
    outputs_dir=paths["outputs"],
    predictions_dir=paths["predictions"],
    id_to_label=id_to_label,
    reset_results=True,
    wandb_config=CONFIG.wandb,
)

display(pd.DataFrame([
    {
        "model_name": row["model_name"],
        "split": row["split"],
        **row["metrics"],
    }
    for row in baseline_results
]))


## 14. Classical TF-IDF Models

Train TF-IDF + Logistic Regression and TF-IDF + Linear SVM. Select the best configuration using validation macro-F1, then evaluate on test.


In [ ]:
RUN_CLASSICAL_MODELS = True

if RUN_CLASSICAL_MODELS:
    classical_results = run_classical_models(
        train_df,
        validation_df,
        test_df,
        outputs_dir=paths["outputs"],
        predictions_dir=paths["predictions"],
        figures_dir=paths["figures"],
        models_dir=paths["models_trained_classical"],
        checkpoints_dir=paths["checkpoints_classical"],
        id_to_label=id_to_label,
        wandb_config=CONFIG.wandb,
    )
    summary_rows = []
    for model_name, result_bundle in classical_results.items():
        test_result = result_bundle["test_result"]
        summary_rows.append({"model_name": model_name, **test_result["metrics"]})
    display(pd.DataFrame(summary_rows).sort_values("macro_f1", ascending=False))
else:
    print("Classical model training skipped because RUN_CLASSICAL_MODELS is False.")


## 15. Transformer Experiment Template

Transformer fine-tuning is disabled by default. Change `RUN_TRANSFORMERS` only when you are ready for a long training run.


In [ ]:
RUN_TRANSFORMERS = False
TRANSFORMER_MAX_LENGTH = 256
TRANSFORMER_BATCH_SIZE = 8
TRANSFORMER_EPOCHS = 3

print(f"RUN_TRANSFORMERS: {RUN_TRANSFORMERS}")
print(f"transformers installed: {transformers_available()}")
print(f"torch.cuda.is_available(): {torch.cuda.is_available()}")
print(f"fp16 will be set to torch.cuda.is_available(): {torch.cuda.is_available()}")

if RUN_TRANSFORMERS and torch.cuda.is_available():
    torch.cuda.empty_cache()
elif RUN_TRANSFORMERS and not torch.cuda.is_available():
    print("WARNING: RUN_TRANSFORMERS is True but CUDA is unavailable. Training may be very slow.")

transformer_results = run_transformer_experiments(
    train_df,
    validation_df,
    test_df,
    outputs_dir=paths["outputs"],
    predictions_dir=paths["predictions"],
    models_dir=paths["models_trained_transformers"],
    checkpoints_dir=paths["checkpoints_transformers"],
    id_to_label=id_to_label,
    wandb_config=CONFIG.wandb,
    run_transformers=RUN_TRANSFORMERS,
    max_length=TRANSFORMER_MAX_LENGTH,
    batch_size=TRANSFORMER_BATCH_SIZE,
    epochs=TRANSFORMER_EPOCHS,
)

if transformer_results:
    display(pd.DataFrame([{ "model_name": row["model_name"], **row["metrics"] } for row in transformer_results]))


## 16. Final Results Comparison

Combine available result rows and prepare comparison figures. This section prepares evidence only.


In [ ]:
comparison = create_final_comparison(paths["outputs"], paths["figures"], wandb_config=CONFIG.wandb)
display(comparison.sort_values("macro_f1", ascending=False))


## 17. Error Analysis Evidence

For the best available model, save confusion-matrix evidence and example predictions. Do not interpret the results here.

TODO: Inspect the saved JSON and figures before writing your own analysis separately.


In [ ]:
error_analysis = create_error_analysis(paths["outputs"], paths["figures"], wandb_config=CONFIG.wandb)
print(f"Best model: {error_analysis['best_model']['model_name']}")
print(f"Saved confusion matrix: {error_analysis['confusion_matrix_path']}")

display(pd.DataFrame(error_analysis["top_confused_label_pairs"]).head(10))
display(pd.DataFrame(error_analysis["correct_prediction_examples"]).head(5))
display(pd.DataFrame(error_analysis["incorrect_prediction_examples"]).head(5))


## 18. CUAD as a Secondary Dataset

CUAD is kept separate from LEDGAR because the datasets were created for different task designs. LEDGAR is already a clause classification dataset with a label assigned to each clause. CUAD was designed for contract review and span extraction: labels are attached to answer spans inside contract contexts.

For this optional branch, CUAD spans are adapted into a clause-level classification format by treating each non-empty annotated answer span as `text` and the associated CUAD question or clause type as `label`. This creates a secondary robustness or external-validation experiment without merging CUAD into the LEDGAR training data.

Limitations: LEDGAR and CUAD differ in annotation style, label taxonomy, source contracts, and original benchmark objective. The comparison table below is therefore descriptive and should not be treated as a direct like-for-like dataset ranking.

In [ ]:
import re
from collections import Counter
from typing import Any

from sklearn.dummy import DummyClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.svm import LinearSVC

CUAD_SCHEMA = ["text", "label", "label_id", "source_dataset", "source_id", "split"]


def clean_legal_text(text: Any) -> str:
    """Normalise whitespace only, preserving legal wording and punctuation."""
    if text is None:
        return ""
    try:
        if pd.isna(text):
            return ""
    except (TypeError, ValueError):
        pass
    return re.sub(r"\s+", " ", str(text)).strip()


def save_jsonl(df: pd.DataFrame, path: Path | str) -> Path:
    """Save a DataFrame to JSONL."""
    output_path = Path(path)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    df.to_json(output_path, orient="records", lines=True, force_ascii=False)
    return output_path


def write_json_file(path: Path | str, payload: Any) -> Path:
    """Write JSON with stable formatting."""
    output_path = Path(path)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    output_path.write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding="utf-8")
    return output_path


def safe_filename(value: str) -> str:
    """Create a filesystem-safe filename component."""
    cleaned = re.sub(r"[^a-zA-Z0-9]+", "_", value.lower()).strip("_")
    return cleaned[:80] or "item"


def infer_split_from_path(path: Path) -> str | None:
    """Infer an official split from a filename when present."""
    name = path.stem.lower()
    if "train" in name:
        return "train"
    if "validation" in name or "valid" in name or "dev" in name or "val" in name:
        return "validation"
    if "test" in name:
        return "test"
    return None


def normalise_split_name(value: Any) -> str:
    """Map common split names to train/validation/test."""
    split = clean_legal_text(value).lower()
    if split in {"valid", "validation", "dev", "val"}:
        return "validation"
    if split in {"train", "training"}:
        return "train"
    if split in {"test", "testing"}:
        return "test"
    return ""


def load_cuad_dataset(path: Path | str) -> list[dict[str, Any]] | None:
    """Load SQuAD-style JSON, JSONL, or CSV CUAD files."""
    base_path = Path(path)
    if not base_path.is_absolute():
        base_path = PROJECT_ROOT / base_path
    base_path = base_path.resolve()

    if not base_path.exists():
        return None

    if base_path.is_file():
        candidates = [base_path]
    else:
        candidates = []
        for name in ["CUAD_v1.json", "cuad_v1.json", "CUAD.json", "cuad.json"]:
            candidate = base_path / name
            if candidate.exists():
                candidates.append(candidate)
        for pattern in ("*.json", "*.jsonl", "*.csv"):
            for candidate in sorted(base_path.glob(pattern)):
                if candidate not in candidates:
                    candidates.append(candidate)

    loaded = []
    for file_path in candidates:
        suffix = file_path.suffix.lower()
        try:
            if suffix == ".json":
                with file_path.open("r", encoding="utf-8") as f:
                    payload = json.load(f)
                loaded.append({"format": "json", "path": file_path, "data": payload, "split": infer_split_from_path(file_path)})
            elif suffix == ".jsonl":
                rows = []
                with file_path.open("r", encoding="utf-8") as f:
                    for line in f:
                        line = line.strip()
                        if line:
                            rows.append(json.loads(line))
                loaded.append({"format": "jsonl", "path": file_path, "data": rows, "split": infer_split_from_path(file_path)})
            elif suffix == ".csv":
                loaded.append({"format": "csv", "path": file_path, "data": pd.read_csv(file_path), "split": infer_split_from_path(file_path)})
        except Exception as exc:
            print(f"WARNING: Could not load CUAD file {file_path}: {type(exc).__name__}: {exc}")

    return loaded or None


def infer_cuad_label_from_qa(qa: dict[str, Any]) -> str:
    """Use explicit clause metadata, QA id, or question text as the label."""
    for key in ("clause_type", "clause_category", "category", "label", "title", "question_title"):
        value = clean_legal_text(qa.get(key))
        if value:
            return value

    qa_id = clean_legal_text(qa.get("id"))
    if "__" in qa_id:
        candidate = qa_id.rsplit("__", 1)[-1].replace("_", " ")
        if candidate:
            return clean_legal_text(candidate)

    question = clean_legal_text(qa.get("question"))
    if not question:
        return ""

    quoted = re.search(r'related to ["“]([^"”]+)["”]', question, flags=re.IGNORECASE)
    if quoted:
        return clean_legal_text(quoted.group(1))

    patterns = [
        r"^highlight the parts? \(if any\) of this contract (?:that|which)?\s*(?:are|is)?\s*(?:related to|relate to|address|describe|concern)\s*",
        r"^highlight the parts? \(if any\) of this contract\s*",
        r"^does this contract (?:include|contain|have)\s*",
    ]
    label = question
    for pattern in patterns:
        candidate = re.sub(pattern, "", question, flags=re.IGNORECASE).strip()
        if candidate and candidate != question:
            label = candidate
            break
    return label.strip(" ?.:") or question


def extract_answers_from_qa(qa: dict[str, Any]) -> list[str]:
    """Return non-empty answer texts from a SQuAD-style QA item."""
    if qa.get("is_impossible") is True:
        return []
    answers = qa.get("answers") or []
    extracted = []
    for answer in answers:
        if isinstance(answer, dict):
            text = clean_legal_text(answer.get("text") or answer.get("answer") or answer.get("span"))
        else:
            text = clean_legal_text(answer)
        if text:
            extracted.append(text)
    return extracted


def extract_squad_style_examples(payload: dict[str, Any], source_path: Path, split_hint: str | None) -> list[dict[str, Any]]:
    """Extract span-labelled examples from SQuAD-style CUAD JSON."""
    records = []
    for doc_idx, document in enumerate(payload.get("data", [])):
        source_id = clean_legal_text(
            document.get("title")
            or document.get("contract_name")
            or document.get("id")
            or f"{source_path.stem}_{doc_idx}"
        )
        paragraphs = document.get("paragraphs") or []
        if not paragraphs and "qas" in document:
            paragraphs = [document]

        for paragraph in paragraphs:
            for qa in paragraph.get("qas") or []:
                label = infer_cuad_label_from_qa(qa)
                if not label:
                    continue
                split = normalise_split_name(qa.get("split")) or split_hint or ""
                for answer_text in extract_answers_from_qa(qa):
                    records.append(
                        {
                            "text": answer_text,
                            "label": label,
                            "label_id": -1,
                            "source_dataset": "CUAD",
                            "source_id": source_id,
                            "split": split,
                        }
                    )
    return records


def parse_possible_span_values(value: Any) -> list[str]:
    """Parse answer span cells from CSV/JSONL rows."""
    if value is None:
        return []
    try:
        if pd.isna(value):
            return []
    except (TypeError, ValueError):
        pass

    if isinstance(value, list):
        values = value
    elif isinstance(value, dict):
        values = [value.get("text") or value.get("answer") or value.get("span") or ""]
    else:
        text = clean_legal_text(value)
        if not text or text in {"[]", "{}"}:
            return []
        try:
            parsed = json.loads(text)
            if isinstance(parsed, list):
                values = parsed
            elif isinstance(parsed, dict):
                values = [parsed.get("text") or parsed.get("answer") or parsed.get("span") or ""]
            else:
                values = [parsed]
        except Exception:
            values = re.split(r"\s*\|\|\|\s*", text)

    cleaned = []
    for item in values:
        if isinstance(item, dict):
            item = item.get("text") or item.get("answer") or item.get("span") or ""
        item_text = clean_legal_text(item)
        if item_text:
            cleaned.append(item_text)
    return cleaned


def extract_tabular_cuad_examples(frame: pd.DataFrame, source_path: Path, split_hint: str | None) -> list[dict[str, Any]]:
    """Extract CUAD examples from direct or wide CSV/JSONL tables."""
    records = []
    columns = list(frame.columns)
    lower_to_column = {column.lower(): column for column in columns}

    text_col = next((lower_to_column[c] for c in ("text", "clause_span", "answer_text", "answer", "clause", "span") if c in lower_to_column), None)
    label_col = next((lower_to_column[c] for c in ("label", "clause_category", "category", "clause_type", "question") if c in lower_to_column), None)
    source_col = next((lower_to_column[c] for c in ("source_id", "contract_name", "document_id", "doc_id", "title", "id") if c in lower_to_column), None)
    split_col = next((lower_to_column[c] for c in ("split", "set", "partition") if c in lower_to_column), None)

    if text_col and label_col:
        for row_idx, row in frame.iterrows():
            label = clean_legal_text(row.get(label_col))
            split = normalise_split_name(row.get(split_col)) if split_col else (split_hint or "")
            source_id = clean_legal_text(row.get(source_col)) if source_col else f"{source_path.stem}_{row_idx}"
            for span in parse_possible_span_values(row.get(text_col)):
                if label and span:
                    records.append({"text": span, "label": label, "label_id": -1, "source_dataset": "CUAD", "source_id": source_id, "split": split})
        return records

    metadata_columns = {column for column in (source_col, split_col) if column}
    metadata_columns.update({"context", "contract", "filename", "file_name"})
    candidate_label_columns = [column for column in columns if column not in metadata_columns]

    for row_idx, row in frame.iterrows():
        split = normalise_split_name(row.get(split_col)) if split_col else (split_hint or "")
        source_id = clean_legal_text(row.get(source_col)) if source_col else f"{source_path.stem}_{row_idx}"
        for column in candidate_label_columns:
            label = clean_legal_text(column)
            for span in parse_possible_span_values(row.get(column)):
                if label and span:
                    records.append({"text": span, "label": label, "label_id": -1, "source_dataset": "CUAD", "source_id": source_id, "split": split})
    return records


def extract_cuad_clause_examples(raw_data: list[dict[str, Any]]) -> pd.DataFrame:
    """Extract CUAD examples into the unified clause-classification schema."""
    records = []
    for item in raw_data:
        file_path = Path(item["path"])
        split_hint = item.get("split")
        payload = item["data"]
        if isinstance(payload, dict) and "data" in payload:
            records.extend(extract_squad_style_examples(payload, file_path, split_hint))
        elif isinstance(payload, list):
            if payload and isinstance(payload[0], dict) and "data" in payload[0]:
                for sub_payload in payload:
                    records.extend(extract_squad_style_examples(sub_payload, file_path, split_hint))
            else:
                records.extend(extract_tabular_cuad_examples(pd.DataFrame(payload), file_path, split_hint))
        elif isinstance(payload, pd.DataFrame):
            records.extend(extract_tabular_cuad_examples(payload, file_path, split_hint))

    df = pd.DataFrame(records, columns=CUAD_SCHEMA)
    if df.empty:
        return df
    df["text"] = df["text"].map(clean_legal_text)
    df["label"] = df["label"].map(clean_legal_text)
    df["source_id"] = df["source_id"].map(clean_legal_text)
    df = df[(df["text"] != "") & (df["label"] != "")].copy()
    df = df.drop_duplicates(subset=["text", "label"]).reset_index(drop=True)
    return df[CUAD_SCHEMA]


def filter_cuad_labels(
    df: pd.DataFrame,
    min_samples: int = CUAD_MIN_SAMPLES_PER_CLASS,
    top_k: int | None = CUAD_TOP_K_LABELS,
) -> tuple[pd.DataFrame, dict[str, int], dict[int, str], pd.Series]:
    """Filter CUAD labels and create integer IDs after filtering."""
    counts = df["label"].value_counts()
    kept_counts = counts[counts >= min_samples]
    if top_k:
        kept_counts = kept_counts.head(top_k)
    kept_labels = kept_counts.index.tolist()
    filtered = df[df["label"].isin(kept_labels)].copy().reset_index(drop=True)
    label_to_id = {label: idx for idx, label in enumerate(kept_labels)}
    id_to_label = {idx: label for label, idx in label_to_id.items()}
    if not filtered.empty:
        filtered["label_id"] = filtered["label"].map(label_to_id).astype(int)
    return filtered[CUAD_SCHEMA], label_to_id, id_to_label, kept_counts


def stratify_or_none(df: pd.DataFrame) -> pd.Series | None:
    """Return labels for stratification only when every class has at least two rows."""
    counts = df["label_id"].value_counts()
    return df["label_id"] if not counts.empty and counts.min() >= 2 else None


def split_cuad_dataset(df: pd.DataFrame, seed: int = SEED) -> dict[str, pd.DataFrame]:
    """Preserve official splits when present, otherwise create a stratified 70/15/15 split."""
    working = df.copy()
    working["split"] = working["split"].map(normalise_split_name)
    official = working[working["split"].isin(["train", "validation", "test"])].copy()

    if not official.empty:
        split_names = set(official["split"])
        if {"train", "validation", "test"}.issubset(split_names):
            return {split: official[official["split"] == split].reset_index(drop=True)[CUAD_SCHEMA] for split in ("train", "validation", "test")}
        if {"train", "test"}.issubset(split_names) and "validation" not in split_names:
            train_only = official[official["split"] == "train"].copy()
            test_df = official[official["split"] == "test"].copy()
            train_df, validation_df = train_test_split(
                train_only,
                test_size=0.1765,
                random_state=seed,
                stratify=stratify_or_none(train_only),
            )
            train_df = train_df.copy()
            validation_df = validation_df.copy()
            test_df = test_df.copy()
            train_df["split"] = "train"
            validation_df["split"] = "validation"
            test_df["split"] = "test"
            return {"train": train_df.reset_index(drop=True)[CUAD_SCHEMA], "validation": validation_df.reset_index(drop=True)[CUAD_SCHEMA], "test": test_df.reset_index(drop=True)[CUAD_SCHEMA]}

    train_df, temp_df = train_test_split(working, test_size=0.30, random_state=seed, stratify=stratify_or_none(working))
    validation_df, test_df = train_test_split(temp_df, test_size=0.50, random_state=seed, stratify=stratify_or_none(temp_df))
    train_df = train_df.copy()
    validation_df = validation_df.copy()
    test_df = test_df.copy()
    train_df["split"] = "train"
    validation_df["split"] = "validation"
    test_df["split"] = "test"
    return {"train": train_df.reset_index(drop=True)[CUAD_SCHEMA], "validation": validation_df.reset_index(drop=True)[CUAD_SCHEMA], "test": test_df.reset_index(drop=True)[CUAD_SCHEMA]}


def save_cuad_processed_outputs(splits: dict[str, pd.DataFrame], label_counts: pd.Series, output_dir: Path | str) -> dict[str, Path]:
    """Save processed CUAD splits and metadata."""
    output_path = Path(output_dir)
    output_path.mkdir(parents=True, exist_ok=True)
    saved = {
        "train": save_jsonl(splits["train"], output_path / "cuad_train.jsonl"),
        "validation": save_jsonl(splits["validation"], output_path / "cuad_validation.jsonl"),
        "test": save_jsonl(splits["test"], output_path / "cuad_test.jsonl"),
    }
    label_names = label_counts.index.tolist()
    label_names_path = output_path / "cuad_label_names.txt"
    label_names_path.write_text("\n".join(label_names) + "\n", encoding="utf-8")
    label_counts_path = write_json_file(output_path / "cuad_label_counts.json", label_counts.astype(int).to_dict())
    saved.update({"label_names": label_names_path, "label_counts": label_counts_path})
    return saved


def create_cuad_eda_outputs(
    splits: dict[str, pd.DataFrame],
    output_dir: Path | str,
    results_dir: Path | str,
    examples_per_label: int = 3,
) -> tuple[dict[str, Any], pd.DataFrame]:
    """Create CUAD EDA tables, plots, and examples."""
    output_path = Path(output_dir)
    plots_dir = Path(results_dir) / "plots"
    output_path.mkdir(parents=True, exist_ok=True)
    plots_dir.mkdir(parents=True, exist_ok=True)

    combined = pd.concat(splits.values(), ignore_index=True)
    combined["word_count"] = combined["text"].str.split().str.len()
    combined["character_count"] = combined["text"].str.len()
    label_counts = combined["label"].value_counts()

    summary = {
        "rows_per_split": {split: int(len(frame)) for split, frame in splits.items()},
        "unique_labels": int(combined["label"].nunique()),
        "label_counts": label_counts.astype(int).to_dict(),
        "word_count_stats": combined["word_count"].describe().to_dict(),
        "character_count_stats": combined["character_count"].describe().to_dict(),
    }
    write_json_file(output_path / "cuad_dataset_summary.json", summary)

    fig, ax = plt.subplots(figsize=(12, 6))
    label_counts.plot(kind="bar", ax=ax)
    ax.set_title("CUAD Class Distribution")
    ax.set_xlabel("Label")
    ax.set_ylabel("Rows")
    ax.tick_params(axis="x", labelrotation=90, labelsize=7)
    fig.tight_layout()
    fig.savefig(plots_dir / "cuad_class_distribution.png", dpi=150, bbox_inches="tight")
    plt.close(fig)

    fig, ax = plt.subplots(figsize=(10, 5))
    combined["word_count"].plot(kind="hist", bins=50, ax=ax)
    ax.set_title("CUAD Text Length Distribution")
    ax.set_xlabel("Word count")
    ax.set_ylabel("Rows")
    fig.tight_layout()
    fig.savefig(plots_dir / "cuad_text_length_distribution.png", dpi=150, bbox_inches="tight")
    plt.close(fig)

    example_rows = []
    for label in label_counts.index:
        examples = combined[combined["label"] == label].head(examples_per_label)
        for row in examples.to_dict(orient="records"):
            example_rows.append({"label": row["label"], "label_id": int(row["label_id"]), "split": row["split"], "text": row["text"]})
    examples_df = pd.DataFrame(example_rows)
    save_jsonl(examples_df, Path(results_dir) / "cuad_example_clauses.jsonl")
    return summary, examples_df


def plot_dataset_confusion_matrix(cm: np.ndarray, labels: list[str], path: Path | str, title: str) -> Path:
    """Save a confusion matrix plot for one dataset/model."""
    output_path = Path(path)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    size = max(8, min(18, len(labels) * 0.55))
    fig, ax = plt.subplots(figsize=(size, size))
    image = ax.imshow(cm, interpolation="nearest", cmap="Blues")
    ax.set_title(title)
    fig.colorbar(image, ax=ax, fraction=0.046, pad=0.04)
    positions = np.arange(len(labels))
    ax.set_xticks(positions)
    ax.set_yticks(positions)
    ax.set_xticklabels(labels, rotation=90, fontsize=7)
    ax.set_yticklabels(labels, fontsize=7)
    ax.set_xlabel("Predicted label")
    ax.set_ylabel("True label")
    fig.tight_layout()
    fig.savefig(output_path, dpi=150, bbox_inches="tight")
    plt.close(fig)
    return output_path


def evaluate_dataset_model(
    dataset_name: str,
    model_name: str,
    y_true: list[int],
    y_pred: list[int],
    id_to_label: dict[int, str],
    results_dir: Path | str,
) -> dict[str, Any]:
    """Compute metrics and save classification report plus confusion matrix."""
    labels = sorted(id_to_label)
    target_names = [id_to_label[label_id] for label_id in labels]
    report = classification_report(y_true, y_pred, labels=labels, target_names=target_names, output_dict=True, zero_division=0)
    cm = confusion_matrix(y_true, y_pred, labels=labels)
    results_path = Path(results_dir)
    report_path = results_path / "classification_reports" / f"{safe_filename(dataset_name)}_{safe_filename(model_name)}_report.json"
    cm_path = results_path / "confusion_matrices" / f"{safe_filename(dataset_name)}_{safe_filename(model_name)}_confusion_matrix.png"
    write_json_file(report_path, report)
    plot_dataset_confusion_matrix(cm, target_names, cm_path, f"{dataset_name.upper()} Confusion Matrix: {model_name}")
    return {
        "dataset": dataset_name,
        "model": model_name,
        "accuracy": accuracy_score(y_true, y_pred),
        "macro_f1": f1_score(y_true, y_pred, labels=labels, average="macro", zero_division=0),
        "weighted_f1": f1_score(y_true, y_pred, labels=labels, average="weighted", zero_division=0),
        "number_of_classes": len(labels),
        "classification_report_path": str(report_path),
        "confusion_matrix_path": str(cm_path),
    }


def build_tfidf_model(model_name: str, config: dict[str, Any]) -> Pipeline:
    """Build a TF-IDF classical model matching the LEDGAR model family."""
    vectorizer = TfidfVectorizer(lowercase=True, ngram_range=config["ngram_range"], max_features=config["max_features"])
    if model_name == "logistic_regression":
        classifier = LogisticRegression(max_iter=1000, class_weight="balanced", random_state=SEED, n_jobs=-1)
    elif model_name == "linear_svm":
        classifier = LinearSVC(class_weight="balanced", random_state=SEED)
    else:
        raise ValueError(f"Unsupported model_name: {model_name}")
    return Pipeline([("tfidf", vectorizer), ("classifier", classifier)])


def run_baseline_experiments_for_dataset(
    dataset_name: str,
    train_df: pd.DataFrame,
    val_df: pd.DataFrame,
    test_df: pd.DataFrame,
    results_dir: Path | str,
) -> pd.DataFrame:
    """Run Majority, TF-IDF Logistic Regression, and TF-IDF Linear SVM on one dataset."""
    results_path = Path(results_dir)
    results_path.mkdir(parents=True, exist_ok=True)
    combined_labels = pd.concat([train_df, val_df, test_df], ignore_index=True)
    id_to_label = {
        int(row["label_id"]): row["label"]
        for row in combined_labels[["label_id", "label"]].drop_duplicates().sort_values("label_id").to_dict(orient="records")
    }

    rows = []
    x_train = train_df["text"].tolist()
    y_train = train_df["label_id"].astype(int).tolist()
    x_val = val_df["text"].tolist()
    y_val = val_df["label_id"].astype(int).tolist()
    x_test = test_df["text"].tolist()
    y_test = test_df["label_id"].astype(int).tolist()

    majority = DummyClassifier(strategy="most_frequent")
    majority.fit(x_train, y_train)
    majority_pred = majority.predict(x_test).astype(int).tolist()
    rows.append(evaluate_dataset_model(dataset_name, "majority_baseline", y_test, majority_pred, id_to_label, results_path))

    grid = [
        {"ngram_range": (1, 1), "max_features": 10000},
        {"ngram_range": (1, 1), "max_features": 30000},
        {"ngram_range": (1, 2), "max_features": 10000},
        {"ngram_range": (1, 2), "max_features": 30000},
    ]
    for model_name in ("logistic_regression", "linear_svm"):
        best_model = None
        best_config = None
        best_macro_f1 = -1.0
        for config in grid:
            model = build_tfidf_model(model_name, config)
            model.fit(x_train, y_train)
            val_pred = model.predict(x_val).astype(int).tolist()
            val_macro_f1 = f1_score(y_val, val_pred, labels=sorted(id_to_label), average="macro", zero_division=0)
            if val_macro_f1 > best_macro_f1:
                best_model = model
                best_config = config
                best_macro_f1 = val_macro_f1

        if best_model is None or best_config is None:
            raise RuntimeError(f"No model was trained for {dataset_name}/{model_name}.")
        test_pred = best_model.predict(x_test).astype(int).tolist()
        result = evaluate_dataset_model(dataset_name, model_name, y_test, test_pred, id_to_label, results_path)
        result["validation_macro_f1"] = best_macro_f1
        result["best_ngram_range"] = str(best_config["ngram_range"])
        result["best_max_features"] = best_config["max_features"]
        rows.append(result)

    results_df = pd.DataFrame(rows)
    results_df.to_csv(results_path / "baseline_results.csv", index=False)
    return results_df

## 19. Load and Preprocess CUAD

This cell runs only when `RUN_CUAD_EXPERIMENT` is `True` and CUAD files are present. If no CUAD files are found, the notebook prints a clear skip message and continues with the LEDGAR outputs.

In [ ]:
cuad_raw_path = PROJECT_ROOT / CUAD_DATA_PATH
cuad_output_path = PROJECT_ROOT / CUAD_OUTPUT_DIR
cuad_results_path = PROJECT_ROOT / CUAD_RESULTS_DIR

cuad_experiment_ran = False
cuad_splits = {}
cuad_results_df = pd.DataFrame()
cuad_summary = {}
cuad_examples_df = pd.DataFrame()

if not RUN_CUAD_EXPERIMENT:
    print("CUAD experiment skipped because RUN_CUAD_EXPERIMENT is False.")
else:
    raw_cuad = load_cuad_dataset(cuad_raw_path)
    if raw_cuad is None:
        print("CUAD experiment skipped because CUAD data files were not found.")
    else:
        print(f"Loaded {len(raw_cuad)} CUAD file(s).")
        cuad_all_df = extract_cuad_clause_examples(raw_cuad)
        print(f"Extracted CUAD span examples after duplicate removal: {len(cuad_all_df)}")

        if cuad_all_df.empty:
            print("CUAD experiment skipped because no non-empty labelled clause spans were extracted.")
        else:
            cuad_filtered_df, cuad_label_to_id, cuad_id_to_label, cuad_label_counts = filter_cuad_labels(
                cuad_all_df,
                min_samples=CUAD_MIN_SAMPLES_PER_CLASS,
                top_k=CUAD_TOP_K_LABELS,
            )
            print(f"CUAD rows after label filtering: {len(cuad_filtered_df)}")
            print(f"CUAD labels after filtering: {len(cuad_label_to_id)}")

            if cuad_filtered_df.empty or len(cuad_label_to_id) < 2:
                print("CUAD experiment skipped because too few labels remained after filtering.")
            else:
                cuad_splits = split_cuad_dataset(cuad_filtered_df, seed=SEED)
                saved_cuad_paths = save_cuad_processed_outputs(cuad_splits, cuad_label_counts, cuad_output_path)
                cuad_summary, cuad_examples_df = create_cuad_eda_outputs(cuad_splits, cuad_output_path, cuad_results_path)
                cuad_experiment_ran = True

                print("Saved CUAD processed files:")
                for name, path in saved_cuad_paths.items():
                    print(f"  {name}: {path}")
                display(pd.DataFrame([
                    {"split": split, "rows": len(frame), "labels": frame["label"].nunique()}
                    for split, frame in cuad_splits.items()
                ]))
                display(pd.DataFrame(cuad_label_counts.rename("count")).reset_index().rename(columns={"index": "label"}))
                display(cuad_examples_df.head(20))

## 20. CUAD Baseline and Classical Models

The CUAD branch uses the same model families as the main LEDGAR classical experiment where possible: Majority baseline, TF-IDF Logistic Regression, and TF-IDF Linear SVM. Results are written separately under `results/cuad/`.

In [ ]:
if not cuad_experiment_ran:
    print("CUAD model training skipped because the CUAD preprocessing branch did not run.")
else:
    cuad_results_df = run_baseline_experiments_for_dataset(
        "cuad",
        cuad_splits["train"],
        cuad_splits["validation"],
        cuad_splits["test"],
        cuad_results_path,
    )
    print(f"Saved CUAD results to: {cuad_results_path / 'baseline_results.csv'}")
    display(cuad_results_df[["dataset", "model", "accuracy", "macro_f1", "weighted_f1", "number_of_classes"]])

## 21. LEDGAR vs CUAD Comparison

This table compares datasets only for matching model families where results are available. It is a descriptive robustness check, not a direct benchmark merge, because CUAD and LEDGAR differ in labels, annotation process, and original task formulation.

In [ ]:
def make_ledgar_dataset_rows(
    ledgar_comparison: pd.DataFrame,
    train_df: pd.DataFrame,
    validation_df: pd.DataFrame,
    test_df: pd.DataFrame,
    id_to_label: dict[int, str],
) -> pd.DataFrame:
    rows = []
    model_column = "model_name" if "model_name" in ledgar_comparison.columns else "model"
    matching_models = {"majority_baseline", "logistic_regression", "linear_svm"}
    for row in ledgar_comparison.to_dict(orient="records"):
        model = row.get(model_column)
        if model not in matching_models:
            continue
        rows.append(
            {
                "dataset": "LEDGAR",
                "model": model,
                "accuracy": row.get("accuracy"),
                "macro_f1": row.get("macro_f1"),
                "weighted_f1": row.get("weighted_f1"),
                "number_of_classes": len(id_to_label),
                "train_size": len(train_df),
                "validation_size": len(validation_df),
                "test_size": len(test_df),
            }
        )
    return pd.DataFrame(rows)


def make_cuad_dataset_rows(cuad_results: pd.DataFrame, splits: dict[str, pd.DataFrame]) -> pd.DataFrame:
    if cuad_results.empty or not splits:
        return pd.DataFrame()
    rows = []
    for row in cuad_results.to_dict(orient="records"):
        rows.append(
            {
                "dataset": "CUAD",
                "model": row.get("model"),
                "accuracy": row.get("accuracy"),
                "macro_f1": row.get("macro_f1"),
                "weighted_f1": row.get("weighted_f1"),
                "number_of_classes": int(row.get("number_of_classes")),
                "train_size": len(splits["train"]),
                "validation_size": len(splits["validation"]),
                "test_size": len(splits["test"]),
            }
        )
    return pd.DataFrame(rows)


ledgar_dataset_rows = make_ledgar_dataset_rows(comparison, train_df, validation_df, test_df, id_to_label)
cuad_dataset_rows = make_cuad_dataset_rows(cuad_results_df, cuad_splits)
ledgar_cuad_comparison = pd.concat([ledgar_dataset_rows, cuad_dataset_rows], ignore_index=True)

ordered_columns = [
    "dataset",
    "model",
    "accuracy",
    "macro_f1",
    "weighted_f1",
    "number_of_classes",
    "train_size",
    "validation_size",
    "test_size",
]
ledgar_cuad_comparison = ledgar_cuad_comparison.reindex(columns=ordered_columns)

if not ledgar_cuad_comparison.empty:
    comparison_output_path = PROJECT_ROOT / CUAD_RESULTS_DIR / "ledgar_cuad_model_comparison.csv"
    comparison_output_path.parent.mkdir(parents=True, exist_ok=True)
    ledgar_cuad_comparison.to_csv(comparison_output_path, index=False)
    display(ledgar_cuad_comparison.sort_values(["dataset", "macro_f1"], ascending=[True, False]))
    print(f"Saved LEDGAR/CUAD comparison to: {comparison_output_path}")
else:
    print("No LEDGAR/CUAD comparison rows were available.")

print("Concise experiment summary")
if not ledgar_dataset_rows.empty:
    ledgar_best = ledgar_dataset_rows.sort_values("macro_f1", ascending=False).iloc[0]
    print(f"Best LEDGAR model: {ledgar_best['model']} (macro-F1={ledgar_best['macro_f1']:.4f})")
else:
    ledgar_best = None
    print("LEDGAR comparison rows were not available.")

if cuad_experiment_ran and not cuad_dataset_rows.empty:
    cuad_best = cuad_dataset_rows.sort_values("macro_f1", ascending=False).iloc[0]
    print(f"Best CUAD model: {cuad_best['model']} (macro-F1={cuad_best['macro_f1']:.4f})")
    if ledgar_best is not None:
        difference = float(cuad_best["macro_f1"]) - float(ledgar_best["macro_f1"])
        if difference < 0:
            print(f"CUAD best macro-F1 is lower than LEDGAR best macro-F1 by {abs(difference):.4f} in this run.")
        elif difference > 0:
            print(f"CUAD best macro-F1 is higher than LEDGAR best macro-F1 by {difference:.4f} in this run.")
        else:
            print("CUAD best macro-F1 matches LEDGAR best macro-F1 in this run.")
else:
    print("CUAD results were not produced in this run.")

print("LEDGAR result rows:")
display(ledgar_dataset_rows)
if cuad_experiment_ran and not cuad_dataset_rows.empty:
    print("CUAD result rows:")
    display(cuad_dataset_rows)